In [ ]:
import sys
sys.path.append("../")

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import os

from npu_yolov8n import get_dataloader
from npu_yolov8n.data import preprocess_dataset
from npu_yolov8n import load_model

from npu_yolov8n.utils import ActivationStatsCollector
from npu_yolov8n.models.blocks.qat_blocks import QConv

from npu_yolov8n.utils import visualize_multiple_tensors
from npu_yolov8n.training import train, evaluate, PruningFTConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

layer_type_map = {
    '0': 'Conv',
    '1': 'Conv',
    '2': 'C2f',
    '3': 'Conv',
    '4': 'C2f',
    '5': 'Conv',
    '6': 'C2f',
    '7': 'Conv',
    '8': 'C2f',
    '9': 'SPPF',
    '10': 'Upsample',
    '11': 'Concat',
    '12': 'C2f',
    '13': 'Upsample',
    '14': 'Concat',
    '15': 'C2f',
    '16': 'Conv',
    '17': 'Concat',
    '18': 'C2f',
    '19': 'Conv',
    '20': 'Concat',
    '21': 'C2f',
    '22': 'Detect'
}

In [ ]:
# load dataset 
train_loader, val_loader, coco_id2label, label2coco_category = get_dataloader(data_root='../../../jongsul/yoloproject/npu_yolov8n/datasets/coco')

In [ ]:
# load quantized model
checkpoint_path = '../checkpoints/qat_fixed.pt' 
loaded_cfg = torch.load(checkpoint_path, device, weights_only=False)['qcfg']
Qmodel = load_model(checkpoint_path, device, model_type='qat', model_qcfg=loaded_cfg)

# Weight Analysis

In [ ]:
sdict = Qmodel.state_dict()
[k.removesuffix('.weight') for k in sdict.keys() if 'weight' in k ]

In [ ]:
def topk_l1_channels_selection(Qmodel, top_ratio=0.3, mult={}):
    result = {}
    for k, v in Qmodel.state_dict().items():
        if 'weight' not in k: # only check weights
            continue
        l1_norm = v.abs().sum(dim=(1, 2, 3))  # size: (out_channels)
        num_channels = l1_norm.numel()
        k_prune = int(num_channels * top_ratio * mult.get(k, 1.0))  # number of channels to prune

        if k_prune <= 0:
            continue

        prune_idx = torch.topk(l1_norm, k_prune, largest=False).indices

        result[k.removesuffix('.conv.weight')] = prune_idx

    return result
l1_result = topk_l1_channels_selection(Qmodel, top_ratio=0.3)


# Activation Analysis

In [ ]:
# activation analysis

# register hook
asc = ActivationStatsCollector(Qmodel, (QConv,), mode='output') 
asc.clear_stats()

poz_stats = {}  # poz analysis

# get activation stats and compute poz
run_batch = 300
loader_iter = iter(val_loader)
for _ in range(run_batch): # run multiple iterations to collect activation statistics
    # forward pass
    with torch.no_grad():
        try:
            images, annotations = next(loader_iter)
        except StopIteration:
            loader_iter = iter(val_loader)
            images, annotations = next(loader_iter)
        inputs, batch = preprocess_dataset(images, annotations, coco_id2label, device) 
        outputs = Qmodel(inputs, inference=True)
    activation_stats = asc.get_stats()

    # poz analysis
    for layer_name, acts in activation_stats.items():
        acts = acts[0] # [0] since we ran only one batch at a time
        batch, channel, h, w = acts.shape
        zeros = (acts == 0).sum(dim=(0, 2, 3)) / (batch*h*w)
        if layer_name in poz_stats:
            poz_stats[layer_name] += zeros
        else:
            poz_stats[layer_name] = zeros

    # clear stats for next iteration
    asc.clear_stats()

# average poz over batches
for layer_name, zeros in poz_stats.items():
    poz_stats[layer_name] = zeros / run_batch  


In [ ]:
# top k poz channels selection
def topk_poz_channels_selection(poz_stats, top_ratio=0.3, mult={}):
    result = {}
    thresholds = {}

    for key, tensor in poz_stats.items():
        k = max(1, int(len(tensor) * top_ratio * mult.get(key, 1.0)))
        topk = torch.topk(tensor, k)

        indices = topk.indices.tolist()
        values = topk.values.tolist()
        threshold = min(values) 

        result[key] = indices
        thresholds[key] = threshold
    return result


result = topk_poz_channels_selection(poz_stats, top_ratio=0.3)

# Pruning Simulation

In [ ]:
# Hook manager for channel masking
class ChannelMaskHookManager:
    """
    Register hooks on specific layers and zero-out selected output channels.

    Args:
        model (nn.Module): target model
        target_layers (dict):
            key   = layer name (as in model.named_modules())
            value = list of output channel indices to zero-out
    """

    def __init__(self, model, target_layers: dict):
        """
        target_layers example:
            {
                "conv1": [0, 3, 7],
                "layer2.1.conv2": [5, 8]
            }
        """
        self.model = model
        self.target_layers = target_layers
        self.hooks = []
        self._register_hooks()

    def _make_hook(self, layer_name, channel_indices):
        def hook(module, input, output):
            """
            output shape: (B, C, H, W) or (B, C)
            channel_indices: list of channels to zero-out
            """
            out = output

            # Mask 생성
            # out is (B, C, H, W)  또는 (B, C)
            mask = torch.ones_like(out)

            # 채널을 0으로 설정
            # mask[:, channel_indices, ...] = 0
            mask.index_fill_(1, torch.tensor(channel_indices, device=out.device), 0)

            # 마스킹 적용
            out = out * mask
            return out

        return hook

    def _register_hooks(self):
        for name, module in self.model.named_modules():
            if name in self.target_layers:
                channel_indices = self.target_layers[name]

                hook_fn = self._make_hook(name, channel_indices)
                h = module.register_forward_hook(hook_fn)

                self.hooks.append(h)

    def remove_hooks(self):
        """Remove every registered hook."""
        for h in self.hooks:
            h.remove()
        self.hooks = []


In [ ]:
# get pruning target
# example:
"""    
    {'model.0': [1, 12, 10, 14],
    'model.1': [7, 19, 14, 17, 13, 4, 23, 0, 22],
    'model.2.cv2': [31, 22, 24, 15, 20, 11, 28, 30, 26],}
"""
def get_pruning_target(result, layer_type_map):
    pruning_target = {}
    for k, v in result.items():
        layer = k.split('.')[1]
        layer_type = layer_type_map[layer]
        if layer_type == 'Conv':
            pruning_target[k] = v
        elif layer_type == 'C2f':
            if k == f'model.{layer}.cv2':
                pruning_target[k] = v # only output of cv2 is pruned
        elif layer_type == 'SPPF':
            if k == f'model.{layer}.cv2':
                pruning_target[k] = v # only output of cv2 is pruned
        elif layer_type == 'Detect':
            pass # no pruning for Detect layer
        else:
            print("Unknown layer type")
    return pruning_target
# pruning_target = get_pruning_target(result, layer_type_map)


In [ ]:
os.makedirs('./sensitivity_analysis', exist_ok=True)


## Weight L1 Norm-based Pruning

In [ ]:
# sensitivity analysis
result = topk_l1_channels_selection(Qmodel, top_ratio=0.3)
pruning_target = get_pruning_target(result, layer_type_map)
eval_results = {}
for k, v in pruning_target.items():

    hook_manager = ChannelMaskHookManager(Qmodel, {k:v})
    eval_results[k] = evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=True)
    hook_manager.remove_hooks()

torch.save(eval_results, "sensitivity_analysis/l1_sensitivity_analysis_1.pt")


In [ ]:
# sensitivity analysis
test_ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
for top_ratio in test_ratios:
    result = topk_l1_channels_selection(Qmodel, top_ratio=top_ratio)
    pruning_target = get_pruning_target(result, layer_type_map)

    eval_results = {}
    for k, v in pruning_target.items():

        hook_manager = ChannelMaskHookManager(Qmodel, {k:v})
        if k in eval_results:
            eval_results[k].append((evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=True), top_ratio))
        else:
            eval_results[k] = [(evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=True), top_ratio)]
        hook_manager.remove_hooks()

torch.save(eval_results, "sensitivity_analysis/l1_sensitivity_analysis_2.pt")


In [ ]:
# sensitivity analysis
test_ratios = [0.05, 0.10, 0.15, 0.20, 0.25]
eval_results = {}
Qmodel = load_model(checkpoint_path, device, model_type='qat', model_qcfg=loaded_cfg)

for top_ratio in test_ratios:
    result = topk_l1_channels_selection(Qmodel, top_ratio=top_ratio)
    pruning_target = get_pruning_target(result, layer_type_map)
    print(f'model with top_ratio {top_ratio} evaluating...')


    hook_manager = ChannelMaskHookManager(Qmodel, pruning_target)
    eval_results[f'{top_ratio}'] = evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=True)
    hook_manager.remove_hooks()

torch.save(eval_results, "sensitivity_analysis/l1_sensitivity_analysis_3.pt")


## POZ-based Pruning

In [ ]:
# sensitivity analysis
result = topk_poz_channels_selection(poz_stats, top_ratio=0.3)
pruning_target = get_pruning_target(result, layer_type_map)
eval_results = {}
for k, v in pruning_target.items():

    hook_manager = ChannelMaskHookManager(Qmodel, {k:v})
    eval_results[k] = evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=True)
    hook_manager.remove_hooks()

torch.save(eval_results, "sensitivity_analysis/poz_sensitivity_analysis_1.pt")


In [ ]:
# sensitivity analysis
test_ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
for top_ratio in test_ratios:
    result = topk_poz_channels_selection(poz_stats, top_ratio=top_ratio)
    pruning_target = get_pruning_target(result, layer_type_map)

    eval_results = {}
    for k, v in pruning_target.items():

        hook_manager = ChannelMaskHookManager(Qmodel, {k:v})
        if k in eval_results:
            eval_results[k].append((evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=True), top_ratio))
        else:
            eval_results[k] = [(evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=True), top_ratio)]
        hook_manager.remove_hooks()

torch.save(eval_results, "sensitivity_analysis/poz_sensitivity_analysis_2.pt")


In [ ]:
# sensitivity analysis
test_ratios = [0.05, 0.10, 0.15, 0.20, 0.25]
eval_results = {}
Qmodel = load_model(checkpoint_path, device, model_type='qat', model_qcfg=loaded_cfg)

for top_ratio in test_ratios:
    result = topk_poz_channels_selection(poz_stats, top_ratio=top_ratio)
    pruning_target = get_pruning_target(result, layer_type_map)
    print(f'model with top_ratio {top_ratio} evaluating...')


    hook_manager = ChannelMaskHookManager(Qmodel, pruning_target)
    eval_results[f'{top_ratio}'] = evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=True)
    hook_manager.remove_hooks()

torch.save(eval_results, "sensitivity_analysis/poz_sensitivity_analysis_3.pt")


# Analysis

## Parameter Size Analysis

In [ ]:
# count parameter sizes per layer
weight_sizes = {}
sdict = Qmodel.state_dict()
for name, param in sdict.items():
    layer = name.split('.')[1]
    if layer not in weight_sizes:
        weight_sizes[layer] = 0
    if 'weight' in name:
        weight_sizes[layer] += param.numel() # weight is 8bit = 1byte
    elif 'bias' in name:
        weight_sizes[layer] += 2 * param.numel() # bias is 16bit = 2bytes
    else:
        print("Unknown parameter type")

keys = [f'{key}_{layer_type_map[key]}' for key in weight_sizes.keys()]
values = list(weight_sizes.values())

plt.figure(figsize=(12, 3))
plt.bar(keys, values)
plt.xlabel('Category', fontsize=8)
plt.ylabel('Weight Size [bytes]', fontsize=8)
plt.title('Weight Sizes Visualization', fontsize=10)

plt.xticks(fontsize=7)
plt.yticks(fontsize=7)

plt.grid(axis='y', linestyle='--')

plt.show()


## Sensitivity Analysis

### Weight L1 Norm Based

In [ ]:
sresult = torch.load('sensitivity_analysis/l1_sensitivity_analysis_1.pt')

criteria = 0.3244628608226776
keys = [f'{key}_{layer_type_map[key.split('.')[1]]}' for key in sresult.keys()]
values = [v['map']/criteria*100 for k, v in sresult.items()]

plt.figure(figsize=(12, 3))
plt.bar(keys, values)
plt.xlabel('Layer', fontsize=10)
plt.ylabel('mAP Drop (%)', fontsize=10)
plt.title('L1 Based Pruning Sensitivity Analysis', fontsize=10)

plt.xticks(fontsize=7, rotation=30)
plt.yticks(fontsize=7)


plt.grid(axis='y', linestyle='--')

plt.show()


### POZ-based Pruning

In [ ]:
sresult = torch.load('sensitivity_analysis/poz_sensitivity_analysis_1.pt')

criteria = 0.3244628608226776
keys = [f'{key}_{layer_type_map[key.split('.')[1]]}' for key in sresult.keys()]
values = [(v['map']/criteria)*100 for k, v in sresult.items()]

plt.figure(figsize=(12, 3))
plt.bar(keys, values)
plt.xlabel('Layer', fontsize=10)
plt.ylabel('mAP Drop (%)', fontsize=10)
plt.title('Poz Based Pruning Sensitivity Analysis', fontsize=10)

plt.xticks(fontsize=7, rotation=30)
plt.yticks(fontsize=7)


plt.grid(axis='y', linestyle='--')

plt.show()


In [ ]:
sresult = torch.load('sensitivity_analysis/poz_sensitivity_analysis_2.pt')

criteria = 0.3244628608226776

keys = [f'{key}_{layer_type_map[key.split(".")[1]]}' for key in sresult.keys()]
num_keys = len(keys)

# 첫 key에서 pruning 튜플 개수 확인
num_ratios = len(next(iter(sresult.values())))  # e.g. 4 ratios

# bar 폭 및 x좌표
bar_width = 0.1
x = np.arange(num_keys)

plt.figure(figsize=(14, 5))

for i in range(num_ratios):
    # 각 key에서 i번째 ratio의 map 값 수집
    values = [(sresult[k][i][0]['map'] / criteria * 100) for k in sresult.keys()]
    pruning_ratios = [sresult[k][i][1] for k in sresult.keys()]
    
    # ratio별 x offset
    offset = (i - num_ratios/2) * bar_width + bar_width/2

    plt.bar(x + offset, values, width=bar_width, label=f'pruning ratio: {pruning_ratios[i]}')

plt.xticks(x, keys, rotation=30, fontsize=7)
plt.ylabel('mAP Drop (%)', fontsize=10)
plt.title('Poz Based Pruning Sensitivity Analysis', fontsize=12)
plt.grid(axis='y', linestyle='--')
plt.legend(title="Pruning Index", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
sresult = torch.load('sensitivity_analysis/poz_sensitivity_analysis_3.pt')

criteria = 0.3244628608226776

keys = [f'{key}' for key in sresult.keys()]
values = [v['map']/criteria*100 for k, v in sresult.items()]

plt.figure(figsize=(3, 3))
plt.bar(keys, values)
plt.xlabel('Pruning Ratio', fontsize=10)
plt.ylabel('mAP Drop (%)', fontsize=10)
plt.title('Poz Based Pruning Sensitivity Analysis', fontsize=10)

plt.xticks(fontsize=7)
plt.yticks(fontsize=7)


plt.grid(axis='y', linestyle='--')

plt.show()


# Pruning And Fine-Tuning

### L1

In [ ]:
# get pruning ratio from sensitivity analysis
import math
sresult = torch.load('sensitivity_analysis/l1_sensitivity_analysis_1.pt')

criteria = 0.3244628608226776
keys = [k.removesuffix('.conv') for k in sresult.keys()]
values = np.array([1/((1-v['map']/criteria)**0.25)*100 for k, v in sresult.items()])
values /= values.max()
pratio_dict = {}
for i in range(len(values)):
    pratio_dict[keys[i]] = values[i]

# load model
Qmodel = load_model(checkpoint_path, device, model_type='qat', model_qcfg=loaded_cfg)

# evaluate initial model without channel masking
evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=False)

# pruning finetuning config
init_ratio = 0.05
step = 0.05
num_steps = 10

# do iterative finetuning with increasing pruning ratio
eval_results = {}
for i in range(num_steps):
    top_ratio = init_ratio + i * step
    print(f"Current pruning ratio: {top_ratio}")

    # get pruning target
    result = topk_l1_channels_selection(Qmodel, top_ratio=top_ratio, mult=pratio_dict)
    pruning_target = get_pruning_target(result, layer_type_map)

    # pruning simulation using channel masking 
    hook_manager = ChannelMaskHookManager(Qmodel, pruning_target)
    train(Qmodel, train_loader, val_loader, coco_id2label, PruningFTConfig)
    eval_results[f'{top_ratio}'] = evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=True)

    # model save
    save_model_name = f"ratio_{top_ratio}_step_{step}_init{init_ratio}"
    os.makedirs('./checkpoints', exist_ok=True)
    ckpt_path = f'./checkpoints/{save_model_name}.pt'
    torch.save({
        'qcfg': loaded_cfg,
        'train_config': PruningFTConfig,
        'model_state_dict': Qmodel.state_dict(),
    }, ckpt_path)
    print(f"Checkpoint saved: {ckpt_path}")

    # remove hooks
    hook_manager.remove_hooks()


### POZ

In [ ]:
# get pruning ratio from sensitivity analysis
import math
sresult = torch.load('sensitivity_analysis/poz_sensitivity_analysis_1.pt')

criteria = 0.3244628608226776
keys = [k.removesuffix('.conv') for k in sresult.keys()]
values = np.array([1/((1-v['map']/criteria)**0.25)*100 for k, v in sresult.items()])
values /= values.max()
pratio_dict = {}
for i in range(len(values)):
    pratio_dict[keys[i]] = values[i]

# load model
Qmodel = load_model(checkpoint_path, device, model_type='qat', model_qcfg=loaded_cfg)

# evaluate initial model without channel masking
evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=False)

# pruning finetuning config
init_ratio = 0.05
step = 0.05
num_steps = 10

# do iterative finetuning with increasing pruning ratio
eval_results = {}
for i in range(num_steps):
    top_ratio = init_ratio + i * step
    print(f"Current pruning ratio: {top_ratio}")

    # get pruning target
    result = topk_poz_channels_selection(poz_stats, top_ratio=top_ratio, mult=pratio_dict)
    pruning_target = get_pruning_target(result, layer_type_map)

    # pruning simulation using channel masking 
    hook_manager = ChannelMaskHookManager(Qmodel, pruning_target)
    evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=False)
    train(Qmodel, train_loader, val_loader, coco_id2label, PruningFTConfig)
    eval_results[f'{top_ratio}'] = evaluate(Qmodel, val_loader, coco_id2label, map_only=True, return_results=True)

    # model save
    save_model_name = f"ratio_{top_ratio}_step_{step}_init{init_ratio}"
    os.makedirs('./checkpoints', exist_ok=True)
    ckpt_path = f'./checkpoints/{save_model_name}.pt'
    torch.save({
        'qcfg': loaded_cfg,
        'train_config': PruningFTConfig,
        'model_state_dict': Qmodel.state_dict(),
    }, ckpt_path)
    print(f"Checkpoint saved: {ckpt_path}")

    # register hook for activation poz analysis
    asc = ActivationStatsCollector(Qmodel, (QConv,), mode='output') 
    asc.clear_stats()
    poz_stats = {}  # poz analysis

    # get activation stats and compute poz
    run_batch = 300
    loader_iter = iter(val_loader)
    for _ in range(run_batch): # run multiple iterations to collect activation statistics
        # forward pass
        with torch.no_grad():
            try:
                images, annotations = next(loader_iter)
            except StopIteration:
                loader_iter = iter(val_loader)
                images, annotations = next(loader_iter)
            inputs, batch = preprocess_dataset(images, annotations, coco_id2label, device) 
            outputs = Qmodel(inputs, inference=True)
        activation_stats = asc.get_stats()

        # poz analysis
        for layer_name, acts in activation_stats.items():
            acts = acts[0] # [0] since we ran only one batch at a time
            batch, channel, h, w = acts.shape
            zeros = (acts == 0).sum(dim=(0, 2, 3)) / (batch*h*w)
            if layer_name in poz_stats:
                poz_stats[layer_name] += zeros
            else:
                poz_stats[layer_name] = zeros

        # clear stats for next iteration
        asc.clear_stats()

    # average poz over batches
    for layer_name, zeros in poz_stats.items():
        poz_stats[layer_name] = zeros / run_batch  

    # remove hooks
    hook_manager.remove_hooks()

    # restore model
    Qmodel = load_model(ckpt_path, device, model_type='qat', model_qcfg=loaded_cfg)

torch.save(eval_results, "poz_pruning_ft_results.pt")
